# Assessment 03 — End-to-End Analytics Challenge

| | |
|---|---|
| **Estimated time** | 30–45 minutes (self-timed) |
| **Skill level tested** | Full workflow: data loading → cleaning → SQL analytics → reporting |
| **Prerequisites** | Assessments 01 & 02 completed; `pandas`, `sqlite3`, optional `tabulate` |

---

## Scenario

> **You are an analyst at Northridge Manufacturing GmbH.**  
> Your manager has asked for a **one-page KPI summary** before the monthly
> business review.  You have two data extracts:
>
> - **Sales extract** — 6 months of order line items, slightly messy (nulls,
>   mixed date formats, cancelled noise)
> - **Materials master** — product/material reference data
>
> Deliverable: a clean summary table covering revenue by region, top materials,
> month-on-month growth, and a dead-stock flag for any materials with no sales.

---

## Workflow (work through tasks 1–5 in sequence)

```
[Task 1] Load raw data
    ↓
[Task 2] Clean & standardise (pandas)
    ↓
[Task 3] SQL analytics (sqlite3 in-memory DB)
    ↓
[Task 4] Assemble KPI summary table
    ↓
[Task 5 - Bonus] Dead-stock detection
```

Run the **Setup** cell first.


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import sqlite3
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

try:
    from tabulate import tabulate
    HAS_TABULATE = True
except ImportError:
    HAS_TABULATE = False

def pretty_print(df, title="", floatfmt=".2f"):
    """Print a DataFrame as a clean table regardless of tabulate availability."""
    if title:
        print(f"\n{'═'*60}")
        print(f"  {title}")
        print('═'*60)
    if HAS_TABULATE:
        print(tabulate(df, headers='keys', tablefmt='github',
                       floatfmt=floatfmt, showindex=False))
    else:
        print(df.to_string(index=False))
    print()

print("pandas :", pd.__version__)
print("tabulate:", "available" if HAS_TABULATE else "not installed (pip install tabulate) — using fallback")
print("\n✓ Setup complete")


---

## Task 1 — Load the Raw Data

The two extracts are embedded as raw CSV strings below.  
Use `pd.read_csv(StringIO(...))` to load each into a DataFrame.

**Expected shapes:**
- `sales_raw` : 40 rows × 9 columns
- `materials_raw` : 15 rows × 5 columns

> Notice the deliberate messiness in `sales_raw` — you will fix it in Task 2.


In [ ]:
%%time

# ── Raw sales extract (intentionally messy) ───────────────────────────────────
SALES_CSV = """order_id,line,matnr,customer,region,order_date,status,qty,unit_price
SO-2024-001,1,MAT-001,CUST-A,NORTH,2024-10-01,DELIVERED,20,128.00
SO-2024-001,2,MAT-003,CUST-A,NORTH,2024-10-01,DELIVERED,5,250.00
SO-2024-002,1,MAT-002,CUST-B,SOUTH,01/10/2024,DELIVERED,30,85.50
SO-2024-002,2,MAT-005,CUST-B,SOUTH,01/10/2024,CANCELLED,10,320.00
SO-2024-003,1,MAT-001,CUST-C,EAST,2024-10-15,DELIVERED,15,130.00
SO-2024-003,2,,CUST-C,EAST,2024-10-15,OPEN,8,
SO-2024-004,1,MAT-004,CUST-D,WEST,2024-10-22,DELIVERED,25,65.00
SO-2024-005,1,MAT-002,CUST-A,NORTH,2024-11-03,DELIVERED,40,87.00
SO-2024-005,2,MAT-006,CUST-A,NORTH,2024-11-03,DELIVERED,12,180.00
SO-2024-006,1,MAT-001,CUST-E,EAST,03/11/2024,DELIVERED,18,128.00
SO-2024-006,2,MAT-003,CUST-E,EAST,03/11/2024,OPEN,3,252.00
SO-2024-007,1,MAT-005,CUST-B,SOUTH,2024-11-20,DELIVERED,20,315.00
SO-2024-007,2,MAT-004,CUST-B,SOUTH,2024-11-20,CANCELLED,15,66.00
SO-2024-008,1,MAT-007,CUST-D,WEST,2024-11-25,DELIVERED,100,12.50
SO-2024-009,1,MAT-001,CUST-C,EAST,2024-12-02,DELIVERED,22,131.00
SO-2024-009,2,MAT-006,CUST-F,NORTH,2024-12-02,DELIVERED,8,183.00
SO-2024-010,1,MAT-002,CUST-A,NORTH,02/12/2024,DELIVERED,35,88.00
SO-2024-010,2,MAT-005,CUST-A,NORTH,02/12/2024,OPEN,12,322.00
SO-2024-011,1,MAT-003,CUST-G,WEST,2024-12-15,DELIVERED,10,248.00
SO-2024-011,2,MAT-004,CUST-G,WEST,2024-12-15,DELIVERED,30,64.00
SO-2024-012,1,MAT-008,CUST-H,SOUTH,2024-12-20,DELIVERED,50,8.75
SO-2024-013,1,MAT-001,CUST-B,SOUTH,2025-01-08,DELIVERED,28,132.00
SO-2024-013,2,MAT-006,CUST-B,SOUTH,2025-01-08,DELIVERED,14,185.00
SO-2025-014,1,MAT-002,CUST-C,EAST,08/01/2025,DELIVERED,42,89.00
SO-2025-014,2,MAT-009,CUST-C,EAST,08/01/2025,CANCELLED,6,95.00
SO-2025-015,1,MAT-003,CUST-D,WEST,2025-01-20,DELIVERED,8,255.00
SO-2025-015,2,MAT-005,CUST-F,NORTH,2025-01-20,OPEN,16,318.00
SO-2025-016,1,MAT-001,CUST-E,EAST,2025-02-05,DELIVERED,24,134.00
SO-2025-016,2,MAT-007,CUST-E,EAST,2025-02-05,DELIVERED,200,12.75
SO-2025-017,1,MAT-004,CUST-A,NORTH,05/02/2025,DELIVERED,35,67.00
SO-2025-017,2,MAT-006,CUST-A,NORTH,05/02/2025,DELIVERED,18,188.00
SO-2025-018,1,MAT-002,CUST-G,WEST,2025-02-18,DELIVERED,50,90.00
SO-2025-018,2,MAT-010,CUST-G,WEST,2025-02-18,OPEN,8,210.00
SO-2025-019,1,MAT-001,CUST-H,SOUTH,2025-03-01,DELIVERED,30,135.00
SO-2025-019,2,MAT-003,CUST-H,SOUTH,2025-03-01,DELIVERED,12,258.00
SO-2025-020,1,MAT-005,CUST-B,SOUTH,01/03/2025,DELIVERED,25,320.00
SO-2025-020,2,MAT-008,CUST-C,EAST,01/03/2025,DELIVERED,150,9.00
SO-2025-021,1,MAT-006,CUST-D,WEST,2025-03-15,OPEN,20,190.00
SO-2025-021,2,MAT-004,CUST-F,NORTH,2025-03-15,DELIVERED,40,68.00
SO-2025-022,1,MAT-011,CUST-A,NORTH,2025-03-20,DELIVERED,10,450.00
"""

# ── Materials master ──────────────────────────────────────────────────────────
MATERIALS_CSV = """matnr,description,material_type,unit,std_price
MAT-001,Finished Unit Alpha,FERT,PC,130.00
MAT-002,Raw Steel Rod,ROH,KG,88.00
MAT-003,Sub-assembly X,HALB,PC,250.00
MAT-004,Aluminium Sheet,ROH,KG,65.00
MAT-005,Finished Unit Beta,FERT,PC,318.00
MAT-006,Finished Unit Gamma,FERT,PC,185.00
MAT-007,Packaging Foam,NLAG,PC,12.50
MAT-008,Cardboard Box,NLAG,PC,8.75
MAT-009,Copper Wire,ROH,M,95.00
MAT-010,Sub-assembly Y,HALB,PC,210.00
MAT-011,Finished Unit Delta,FERT,PC,450.00
MAT-012,Steel Flange,ROH,PC,42.00
MAT-013,Sub-assembly Z,HALB,PC,135.00
MAT-014,Packaging Tape,NLAG,M,1.20
MAT-015,Rubber Seal,ROH,PC,5.60
"""

# ── YOUR CODE HERE ─────────────────────────────────────────────────────────────
# Load both CSVs into DataFrames called sales_raw and materials_raw
sales_raw     = None  # replace with pd.read_csv(StringIO(SALES_CSV))
materials_raw = None  # replace with pd.read_csv(StringIO(MATERIALS_CSV))

# ── Validation ────────────────────────────────────────────────────────────────
if sales_raw is not None and materials_raw is not None:
    print(f"sales_raw     shape: {sales_raw.shape}   expected: (40, 9)")
    print(f"materials_raw shape: {materials_raw.shape}  expected: (15, 5)")
    print("\nFirst 3 rows of sales_raw:")
    print(sales_raw.head(3).to_string(index=False))


<details>
<summary>💡 Hint — Task 1</summary>

```python
sales_raw     = pd.read_csv(StringIO(SALES_CSV))
materials_raw = pd.read_csv(StringIO(MATERIALS_CSV))
```

</details>


---

## Task 2 — Clean & Standardise

The raw sales data has several quality issues to fix:

| # | Issue | Action |
|---|-------|--------|
| A | Mixed date formats (`YYYY-MM-DD` and `DD/MM/YYYY`) | Parse to `datetime` |
| B | Rows where `matnr` is null | Drop those rows |
| C | `unit_price` is null for some rows | Drop those rows |
| D | `CANCELLED` orders should be excluded from revenue analysis | Filter them out |

After cleaning, you should have a `sales_clean` DataFrame with:
- All dates in `datetime` format
- No nulls in `matnr` or `unit_price`
- No CANCELLED rows
- A new column `revenue = qty * unit_price`

**Expected rows after cleaning:** approximately 28–30 (non-null, non-cancelled,
non-OPEN with missing price).


In [ ]:
%%time

def clean_sales(df):
    """
    Clean the raw sales DataFrame.
    Returns cleaned copy with a revenue column added.
    """
    df = df.copy()

    # YOUR CODE HERE
    # A. Parse mixed date formats:
    #    pd.to_datetime(df['order_date'], dayfirst=True) handles both formats
    #    when dayfirst=True — but test carefully, or use errors='coerce' + fillna

    # B. Drop rows where matnr is null

    # C. Drop rows where unit_price is null

    # D. Remove CANCELLED orders

    # E. Add revenue column

    return df


sales_clean = clean_sales(sales_raw) if sales_raw is not None else None

if sales_clean is not None:
    print(f"Rows after cleaning : {len(sales_clean)}  (from {len(sales_raw)})")
    print(f"Null matnr          : {sales_clean['matnr'].isna().sum()}")
    print(f"Null unit_price     : {sales_clean['unit_price'].isna().sum()}")
    print(f"Cancelled rows      : {(sales_clean['status'] == 'CANCELLED').sum()}")
    print(f"Columns             : {sales_clean.columns.tolist()}")
    print("\nDate dtype:", sales_clean['order_date'].dtype)


<details>
<summary>💡 Hint — Task 2</summary>

```python
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['matnr', 'unit_price'])
df = df[df['status'] != 'CANCELLED']
df['revenue'] = df['qty'] * df['unit_price']
```

`dayfirst=True` tells pandas to try DD/MM/YYYY first, which handles
the mixed formats in this dataset.  `errors='coerce'` turns truly
unparseable values into NaT instead of raising an exception.

</details>


---

## Task 3 — SQL Analytics via sqlite3

Load the clean DataFrames into an in-memory SQLite database, then write
**three SQL queries**:

| Query | Metric |
|-------|--------|
| 3A | Total revenue by region (DELIVERED only) |
| 3B | Top 5 materials by total revenue (any non-cancelled status) |
| 3C | Month-on-month revenue for the last 6 months (DELIVERED only) |

For 3C, also calculate the MoM growth percentage.

**Note:** Use the `sales_clean` DataFrame — cancelled orders are already removed.


In [ ]:
%%time

if sales_clean is not None and materials_raw is not None:
    # ── Load DataFrames into SQLite ───────────────────────────────────────────
    conn3 = sqlite3.connect(":memory:")
    sales_clean.to_sql("sales", conn3, if_exists="replace", index=False)
    materials_raw.to_sql("materials", conn3, if_exists="replace", index=False)

    # ── 3A: Revenue by region (DELIVERED only) ────────────────────────────────
    query_3a = """
    -- YOUR SQL HERE
    -- Table: sales
    -- Filter: status = 'DELIVERED'
    -- Group by: region
    -- Output: region, total_revenue (sorted descending)
    SELECT 'placeholder' AS region, 0 AS total_revenue WHERE 1=0
    """

    # ── 3B: Top 5 materials by total revenue ──────────────────────────────────
    query_3b = """
    -- YOUR SQL HERE
    -- Join sales with materials on matnr
    -- Group by: matnr + description
    -- Output: matnr, description, total_revenue, order_lines
    -- Limit: top 5
    SELECT 'placeholder' AS matnr, 'x' AS description, 0 AS total_revenue, 0 AS order_lines
    WHERE 1=0
    """

    # ── 3C: MoM revenue (last 6 months, DELIVERED) ───────────────────────────
    query_3c = """
    -- YOUR SQL HERE
    -- Use STRFTIME('%Y-%m', order_date) to extract month
    -- Calculate: month, monthly_revenue
    -- Add MoM growth % using LAG() window function
    -- Filter to last 6 months  (WHERE order_date >= DATE('now', '-6 months'))
    -- Sort by month ascending
    SELECT 'placeholder' AS month, 0 AS monthly_revenue, NULL AS mom_growth_pct
    WHERE 1=0
    """

    # ── Run all three ─────────────────────────────────────────────────────────
    df_3a = pd.read_sql_query(query_3a, conn3)
    df_3b = pd.read_sql_query(query_3b, conn3)
    df_3c = pd.read_sql_query(query_3c, conn3)

    pretty_print(df_3a, "3A: Revenue by Region")
    pretty_print(df_3b, "3B: Top 5 Materials by Revenue")
    pretty_print(df_3c, "3C: Month-on-Month Revenue (last 6 months)")


<details>
<summary>💡 Hint — Task 3</summary>

**3A:**
```sql
SELECT region,
       ROUND(SUM(revenue), 2) AS total_revenue
FROM sales
WHERE status = 'DELIVERED'
GROUP BY region
ORDER BY total_revenue DESC;
```

**3B:**
```sql
SELECT s.matnr,
       m.description,
       ROUND(SUM(s.revenue), 2)  AS total_revenue,
       COUNT(*)                   AS order_lines
FROM sales s
LEFT JOIN materials m ON s.matnr = m.matnr
GROUP BY s.matnr, m.description
ORDER BY total_revenue DESC
LIMIT 5;
```

**3C (MoM with LAG):**
```sql
WITH monthly AS (
    SELECT STRFTIME('%Y-%m', order_date) AS month,
           ROUND(SUM(revenue), 2)        AS monthly_revenue
    FROM sales
    WHERE status = 'DELIVERED'
      AND order_date >= DATE('now', '-6 months')
    GROUP BY month
)
SELECT
    month,
    monthly_revenue,
    ROUND(100.0 * (monthly_revenue - LAG(monthly_revenue) OVER (ORDER BY month))
          / NULLIF(LAG(monthly_revenue) OVER (ORDER BY month), 0), 1) AS mom_growth_pct
FROM monthly
ORDER BY month;
```

</details>


---

## Task 4 — Assemble the One-Page KPI Summary

Using the DataFrames produced in Tasks 2 and 3, print a **structured text
summary** that your manager could read in a terminal or paste into an email.

Required sections:
1. **Overall KPIs** — total revenue, total orders, average order value
2. **Revenue by Region** (from 3A)
3. **Top 5 Materials** (from 3B)
4. **MoM Revenue Trend** (from 3C)

Use `pretty_print()` (defined in the setup cell) for the tables.


In [ ]:
%%time

if sales_clean is not None:
    print("=" * 60)
    print("  NORTHRIDGE MANUFACTURING — SALES KPI SUMMARY")
    print("  Period: Oct 2024 – Mar 2025  |  Status: DELIVERED")
    print("=" * 60)

    # ── YOUR CODE HERE ─────────────────────────────────────────────────────────
    # Section 1: Overall KPIs
    # Hint:
    #   delivered = sales_clean[sales_clean['status'] == 'DELIVERED']
    #   total_rev = delivered['revenue'].sum()
    #   total_orders = delivered['order_id'].nunique()
    #   avg_order_val = total_rev / total_orders

    # Section 2: pretty_print(df_3a, "Revenue by Region")
    # Section 3: pretty_print(df_3b, "Top 5 Materials")
    # Section 4: pretty_print(df_3c, "MoM Revenue Trend")

    print("\n[YOUR SUMMARY OUTPUT GOES HERE]")


<details>
<summary>💡 Hint — Task 4</summary>

```python
delivered = sales_clean[sales_clean['status'] == 'DELIVERED']
total_rev     = delivered['revenue'].sum()
total_orders  = delivered['order_id'].nunique()
avg_order_val = total_rev / total_orders

print(f"\n  Total Revenue     : €{total_rev:>12,.2f}")
print(f"  Total Orders      : {total_orders:>12,}")
print(f"  Avg Order Value   : €{avg_order_val:>12,.2f}")

pretty_print(df_3a, "Revenue by Region")
pretty_print(df_3b, "Top 5 Materials by Revenue")
pretty_print(df_3c, "Month-on-Month Revenue Trend")
```

</details>


---

## Task 5 (Bonus) — Dead-Stock Detection

**Dead stock** = materials that exist in the materials master but appear in
**zero** sales order lines (across the entire cleaned dataset, all statuses).

**Task:**  
Find all materials in `materials_raw` that have no matching `matnr` in
`sales_clean`.

Return a DataFrame with columns: `matnr`, `description`, `material_type`,
`std_price`

> In SAP terminology, this flags materials that may need an inventory
> write-down review or obsolescence assessment (MM/CO process).

*Tests: anti-join pattern — `~df.isin()` or left-join with null filter.*


In [ ]:
%%time

if sales_clean is not None and materials_raw is not None:
    # ── YOUR CODE HERE ─────────────────────────────────────────────────────────
    # Approach A (pandas): anti-join using isin
    #   sold_matnrs = set(sales_clean['matnr'].dropna())
    #   dead_stock = materials_raw[~materials_raw['matnr'].isin(sold_matnrs)]

    # Approach B (SQL): LEFT JOIN + WHERE IS NULL
    #   SELECT m.* FROM materials m
    #   LEFT JOIN sales s ON m.matnr = s.matnr
    #   WHERE s.matnr IS NULL

    dead_stock = None  # replace with your result

    if dead_stock is not None:
        pretty_print(
            dead_stock[['matnr','description','material_type','std_price']],
            f"Dead Stock ({len(dead_stock)} materials with zero sales)"
        )


<details>
<summary>💡 Hint — Task 5</summary>

**pandas anti-join:**
```python
sold = set(sales_clean['matnr'].dropna())
dead_stock = materials_raw[~materials_raw['matnr'].isin(sold)].copy()
```

**SQL LEFT JOIN anti-join:**
```sql
SELECT m.matnr, m.description, m.material_type, m.std_price
FROM materials m
LEFT JOIN sales s ON m.matnr = s.matnr
WHERE s.matnr IS NULL
ORDER BY m.matnr;
```

Both approaches are idiomatic.  The SQL version is often more readable when
the logic gets more complex (multiple join conditions, additional filters).

</details>


---

## Self-Assessment Questions

Reflect honestly — this is for your own calibration, not grading.

---

### Time & Fluency

| Question | Yes | Partially | No |
|----------|:---:|:---------:|:--:|
| Completed all 5 tasks within 45 minutes | ☐ | ☐ | ☐ |
| Completed within 30 minutes | ☐ | — | ☐ |
| Needed to look up pandas syntax (groupby, merge, etc.) | ☐ | ☐ | ☐ |
| Needed to look up SQL syntax (CTE, LAG, anti-join) | ☐ | ☐ | ☐ |

---

### Code Quality

| Question | Yes | Partially | No |
|----------|:---:|:---------:|:--:|
| Task 2 cleaning: used vectorised ops (no loops) | ☐ | ☐ | ☐ |
| Task 3C: used SQL window function (LAG) rather than Python post-processing | ☐ | ☐ | ☐ |
| Task 5: used anti-join pattern rather than iterating over rows | ☐ | ☐ | ☐ |
| Output is clean and readable without extra debugging output | ☐ | ☐ | ☐ |

---

### Interpretation

| Result pattern | Meaning |
|----------------|---------|
| All tasks, ≤ 30 min, no lookups | **Production-ready** — move to advanced modules (dbt, Spark, Power BI DAX) |
| All tasks, 30–45 min, 1–2 lookups | **Strong refresher level** — a week of practice will close the gap |
| Tasks 1–3 complete, struggled with 4–5 | **Good foundation** — focus on SQL window functions and pandas merge patterns |
| Needed hints for Tasks 2–3 | **Structured refresher needed** — complete Modules 01–03 of this bootcamp first |

---

> **End of Assessment — Northridge Manufacturing KPI Challenge**  
> Save this notebook with your completed cells as a record of your starting point.
